# 音楽音源分離モデルの損失関数

音楽音源分離モデルの学習に用いられている損失関数を纏めます。

## 時間域L1/MSE損失

推定・目標分離結果の時間域信号（波形）の誤差の絶対値、あるいは二乗。最も基本的な損失関数で、比較的学習が安定しやすく扱いやすいです。

$$
L1_{time} = \left|y-\tilde{y}\right|
$$

$$
L2_{time} = \left|y-\tilde{y}\right|^2
$$

一方、高周波成分誤差に対する学習重みが低く、高域やアタック成分の聴覚的な再現度が不足しやすいため、ほかの損失関数と併用するケースがほとんどです。


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

estimated_wav = torch.randn(1, 4, 44100*6)  # (batch, sources, samples)
target_wav = torch.randn(1, 4, 44100*6)

l1_loss = F.l1_loss(estimated_wav, target_wav)
mse_loss = F.mse_loss(estimated_wav, target_wav)

# 時間域SDR損失

音源分離の標準的な評価指標であるSDRを損失関数として使います。

$$
SDR_{time} = -10\log_{10}\left(\frac{\|y\|^2}{\|y-\tilde{y}\|^2}\right)
$$

ただしそのままSDRを使うことは少なく、推定波形の全体的なスケール誤差を無視する**SI(scale-invariant)-SDR**と、スケール誤差にもペナルティを課す**SD(scale-dependent)-SDR**がよく使われます[^sdr2019]。

SI-SDRを計算するには、まず推定波形との差分信号の二乗誤差が最小になるようなスケーリング係数$\alpha=\frac{\tilde{y}^Ty}{\left|y\right|^2}$で目標波形をリスケールした後、目標波形と推定波形のSDRを計算します。

$$
SISDR_{time} = -10\log_{10}\left(\frac{\|\alpha y\|^2}{\|\alpha y-\tilde{y}\|^2}\right)
$$

SD-SDRは、SDRの分子の目標信号のみリスケールします。式を変形すると、SNRにスケーリング係数$\alpha$のデシベル値を足しただけの形になります。

$$
SDSDR_{time} = -10\log_{10}\left(\frac{\|\alpha y\|^2}{\|y-\tilde{y}\|^2}\right)=SDR_{time}-10\log_{10}\alpha^2
$$

この損失は**推定波形のスケールが目標を下回るような誤差には敏感である一方、逆に目標を上回るような誤差には鈍い**という特徴があります（$\alpha$が1を上回ると、$10\log_{10}\alpha^2
$の伸びが鈍くなるため）。これに対処するため、実際のSDSDR損失は$max\left(SDR_{time},SDSDR_{time}\right)$を使います。

[^sdr2019]: Jonathan Le Roux et al., "SDR – Half-baked or Well Done?", ICASSP, 2019. https://ieeexplore.ieee.org/document/8683855

In [6]:
def sisdr_time_loss(estimated_wav, target_wav):
    # スケーリング係数を計算
    scale = torch.sum(estimated_wav * target_wav, dim=-1, keepdim=True) / torch.sum(target_wav ** 2, dim=-1, keepdim=True)
    # ターゲット信号をリスケール
    projection = scale * target_wav
    # スケールされた信号と推定信号の差分を計算。
    noise = estimated_wav - projection
    # Compute the SI-SDR loss
    sisdr = 10 * torch.log10(torch.sum(projection ** 2, dim=-1) / torch.sum(noise ** 2, dim=-1))
    return -sisdr.mean() 

def sdsdr_time_loss(estimated_wav, target_wav):
    # SNRを計算
    snr = torch.sum(target_wav ** 2, dim=-1) / torch.sum((estimated_wav - target_wav) ** 2, dim=-1)
    snr = 10 * torch.log10(snr)
    # スケーリング係数を計算
    scale = torch.sum(estimated_wav * target_wav, dim=-1, keepdim=True) / torch.sum(target_wav ** 2, dim=-1, keepdim=True)
    scale_db = 10 * torch.log10(scale)
    # SDSDRを計算
    scale_db[scale_db > 0] = 0  # スケーリングが正の場合は0に設定
    sdsdr = snr + scale_db
    return -sdsdr.mean() 

sisdr_loss = sisdr_time_loss(estimated_wav, target_wav)
sdsdr_loss = sdsdr_time_loss(estimated_wav, target_wav)

# Robust quantile-masked MSE

時間域のMSE損失をロバスト化した損失関数です。誤差の最大値を求め、その最大値の一定の分位数（quantile）を上回るものを誤差評価から除外します[^msst2026]。

外れ値や難しいサンプルに引きずられにくくし、学習の安定性が向上するとされています。

[^msst2026]: Roman Solovyev et al., "Music-Source-Separation-Training (MSST): A Unified Framework for Training and Evaluating Music Demixing Models", 2026. https://arxiv.org/abs/2607.23395

In [7]:
class RobustQuantileMaskedMSELoss(nn.Module):
    def __init__(self, quantile=0.9):
        super(RobustQuantileMaskedMSELoss, self).__init__()
        self.quantile = quantile

    def forward(self, estimated_wav, target_wav):
        mse_loss = F.mse_loss(estimated_wav, target_wav, reduction='none')  # (batch, sources, samples)
        quantile = torch.quantile(mse_loss, self.quantile, dim=2, keepdim=True)
        quantile_masked_mse_loss = mse_loss * (mse_loss <= quantile).float()
        return quantile_masked_mse_loss.mean()

robust_quantile_masked_mse_loss = RobustQuantileMaskedMSELoss()(estimated_wav, target_wav)

## マルチ解像度STFT損失

推定・目標分離結果の時間域信号を、複数の窓長やステップサイズでSTFTスペクトログラムに変換し、スペクトログラム間の誤差を同時に評価します。

STFT誤差は時間域の誤差よりも高周波成分の誤差を公平に扱うため、高域やアタック成分の再現品質を改善できます。時間域の誤差よりも、STFT誤差のほうが主観的評価の良し悪しとの相関度が高いとも言われています[^lossfunc2022]。

マルチ解像度STFT損失は、更に長い窓と短い窓のSTFTを併用することで、損失評価の時間分解能と周波数分解能を両立させることができます。

また、スペクトログラム間誤差の計算方法もいくつかの選択肢があります。
- 複素スペクトログラムの振幅の誤差のみ計算する。位相の誤差は無視するので、時間域L1損失など位相も考慮する損失と併用する必要があります。
- 複素スペクトログラムを振幅・位相スペクトログラムに変換してからそれぞれ誤差を計算する。
- 複素スペクトログラム同士の誤差を直接計算する。複素数同士のL1損失は、複素空間内の**ユークリッド距離**（L2ノルム）です。
- 複素スペクトログラムの実部と虚部の誤差を別々に計算し加算する。L1損失を計算した場合、計算結果は複素空間内のユークリッド距離ではなく**マンハッタン距離**になります。

[^lossfunc2022]: E. Gusó et al, "On Loss Functions and Evaluation Metrics for Music Source Separation," ICASSP 2022. https://ieeexplore.ieee.org/document/9746530

In [8]:
class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512], win_lengths=[512, 1024, 2048], mode='magnitude'):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.mode = mode

    def forward(self, estimated_wav, target_wav):
        total_loss = 0.0
        for fft_size, hop_size, win_length in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            estimated_stft = torch.stft(estimated_wav.squeeze(0), n_fft=fft_size, hop_length=hop_size, win_length=win_length, return_complex=True)
            target_stft = torch.stft(target_wav.squeeze(0), n_fft=fft_size, hop_length=hop_size, win_length=win_length, return_complex=True)
            if self.mode == 'magnitude':
                # 振幅の誤差のみ
                total_loss += F.l1_loss(torch.abs(estimated_stft), torch.abs(target_stft))
            elif self.mode == 'magphase':
                # 振幅と位相の誤差
                estimated_mag = torch.abs(estimated_stft)
                target_mag = torch.abs(target_stft)
                estimated_phase = torch.angle(estimated_stft)
                target_phase = torch.angle(target_stft)
                total_loss += F.l1_loss(estimated_mag, target_mag) + F.l1_loss(estimated_phase, target_phase)
            elif self.mode == 'complex':
                # 複素数の誤差
                total_loss += F.l1_loss(estimated_stft, target_stft)
            elif self.mode == 'realimag':
                # 実部と虚部の誤差
                total_loss += F.l1_loss(torch.view_as_real(estimated_stft), torch.view_as_real(target_stft))
        return total_loss / len(self.fft_sizes)
    
multi_stft_loss = MultiResolutionSTFTLoss()(estimated_wav, target_wav)